# CRS v5 Fast Dry Run (P0 / P1 / P2 / P3 + ResNet-50 Zoom + YOLO Snap)
Runs in ~20–25 min on Kaggle T4 (10 logs, 120 steps). Verifies singleton traffic-light questions, YOLO snap (`Snap > 0`), `Hints > 0`, `center_hit`, `best.pt` early-stopping, and P0–P3 comparison chart.

In [ ]:
!pip install -q "transformers>=4.57" peft pyyaml certifi matplotlib ultralytics
!pip uninstall -y -q torchao
import os, glob, shutil, torch
hits = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/run.py', recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), 'crs'))]
assert hits, 'crs_road dataset not attached'
DST = '/kaggle/working/crs_road'
if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(hits[0], DST)
%cd /kaggle/working/crs_road
LOGS, FR, OUT = '/kaggle/working/CRS_Logs', '/kaggle/working/av2_frames', '/kaggle/working/dry_real_v5'
NLOGS, STEPS = 10, 120
print("Loaded:", hits[0], "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
!python download_logs.py --out {LOGS} --limit {NLOGS}
import glob, os, subprocess
from concurrent.futures import ThreadPoolExecutor
ids = sorted(os.path.basename(f)[:-5] for f in glob.glob(f'{LOGS}/annotations/*.json'))
def fetch(lid):
    if os.path.exists(f'{FR}/{lid}/frames.json'): return lid[:8] + ' skip'
    r = subprocess.run(['python', 'fetch_av2_frames.py', '--log', lid, '--meta', f'{LOGS}/metadata', '--out', FR, '--workers', '8'], capture_output=True, text=True, check=True)
    return lid[:8] + ' ' + (r.stdout.strip().splitlines() or ['ok'])[-1]
with ThreadPoolExecutor(3) as ex:
    for m in ex.map(fetch, ids): print(m)

In [ ]:
!python run.py dryrun --mode real --graphs {LOGS}/annotations --meta {LOGS}/metadata --frames {FR} --out {OUT} \
  --nlogs {NLOGS} --max_per_log 80 --dtype fp16 --grad_ckpt --img_side 224 --lora_r 8 \
  --cnn_arch resnet50 --cnn_epochs 5 --steps {STEPS} --batch 4 --eval_limit 40 --use_det

In [ ]:
!python check_real.py --scenes {OUT}/data/scenes --out {OUT}/overlays --n 3
from IPython.display import Image, display
display(Image(filename=f'{OUT}/preds/p3_comparison.png'))